# Day 68 — Evaluation: Retail Demand Predictor Mini-Project

This notebook covers Comparative Model Evaluation (MAE, RMSE, $R^2$), Residual Error Distribution Analysis, Prediction Uncertainty Quantification (95% Confidence Ranges), and Inventory Risk Management

## Validation goals
- Conduct a side-by-side comparative performance review between the Linear Regression baseline and the Random Forest regressor using MAE, RMSE, and $R^2$
- Analyze residual error distributions to verify zero-mean behavior and diagnose potential heteroscedasticity across demand volumes
- Quantify forecast uncertainty by constructing 95% prediction intervals using the standard deviation of test residuals ($\pm 1.96 \times \sigma_{\text{residuals}}$) with a zero-demand floor
- Export finalized evaluated forecasts with lower and upper confidence bounds to `retail_demand_final_evaluated.csv`

In production software engineering, building a model is only half the battle. You must rigorously evaluate its error distribution, compare it against your baseline, and—crucially for our retail project—calculate confidence ranges so inventory managers know the uncertainty bounds around every demand forecast.

### 1. Comparative Performance Review

Let's review what our metrics reveal when we compare our linear baseline against our non-linear Random Forest. In retail forecasting:
- MAE (Mean Absolute Error): Tells us the average unit error (e.g., how many units off our weekly forecast is).
- RMSE (Root Mean Squared Error): Heavily penalizes large stockout or overstock miscalculations.
- $R^2$ (Coefficient of Determination): Measures the proportion of variance explained by our features.

Typically, while a linear regression baseline might capture broad linear trends ($R^2 \approx 0.60\text{--}0.70$), the Random Forest breaks past that ceiling by capturing complex interactions (e.g., how festive multipliers and transport costs combine), pushing $R^2$ significantly higher.

### 2. Analyzing Residuals & Error Distribution

A model evaluation is incomplete if you only look at aggregate numbers. You must analyze residuals ($y_{\text{actual}} - y_{\text{predicted}}$):
- Zero-Mean Error: Ideally, the average of your residuals should be close to zero, meaning your model doesn't systematically overpredict or underpredict.
- Heteroscedasticity: Check whether your model makes larger errors on high-demand items (like fast-moving consumer goods) compared to low-demand items. If errors scale with volume, log-transforming the target variable in future iterations is a standard remedy.

### 3. Quantifying Uncertainty: Generating Confidence / Range

Recall our project requirements from Day 64:
    - Output: expected demand, confidence/range, important factors

In supply chain management, telling a warehouse manager in Lagos or Kano "You will sell exactly 142 units" is risky. If demand spikes to 180, they run out of stock (stockout loss). If demand drops to 100, excess inventory ties up working capital.

Instead, we provide a Prediction Interval (Range). Using the standard deviation of our model's test residuals ($\sigma_{\text{residuals}}$), we can construct a 95% confidence range around any point prediction:$$\text{Lower Bound} = \hat{y} - (1.96 \times \text{RMSE})$$$$\text{Upper Bound} = \hat{y} + (1.96 \times \text{RMSE})$$

### 4. Implementing Comprehensive Evaluation in Python

Let's write a clean script that loads our saved predictions, computes comparative metrics for both models, calculates residual statistics, and generates a structured demand report with confidence ranges.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load the shared predictions CSV from Day 66 & 67
predictions_df = pd.read_csv("retail_demand_predictions.csv")

y_true = predictions_df['actual_demand']
y_pred_baseline = predictions_df['baseline_pred']
y_pred_rf = predictions_df['random_forest_pred']

# 2. Compute Metrics for Baseline (Linear Regression)
base_mae = mean_absolute_error(y_true, y_pred_baseline)
base_rmse = np.sqrt(mean_squared_error(y_true, y_pred_baseline))
base_r2 = r2_score(y_true, y_pred_baseline)

# 3. Compute Metrics for Improved Model (Random Forest)
rf_mae = mean_absolute_error(y_true, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_true, y_pred_rf))
rf_r2 = r2_score(y_true, y_pred_rf)

print("==================================================")
print("     DAY 68: MODEL EVALUATION & COMPARISON REPORT")
print("==================================================\n")

print("--- Baseline Model (Linear Regression) ---")
print(f"MAE:  {base_mae:.2f} units")
print(f"RMSE: {base_rmse:.2f} units")
print(f"R²:   {base_r2:.4f}\n")

print("--- Improved Model (Random Forest) ---")
print(f"MAE:  {rf_mae:.2f} units")
print(f"RMSE: {rf_rmse:.2f} units")
print(f"R²:   {rf_r2:.4f}\n")

# 4. Calculate Prediction Confidence Intervals (Using Random Forest Residuals)
# 95% confidence multiplier is approximately 1.96 * RMSE
residuals = y_true - y_pred_rf
residual_std = np.std(residuals)
confidence_margin = 1.96 * residual_std

# Add prediction range columns to our dataframe for review
predictions_df['rf_lower_bound'] = np.maximum(0, (y_pred_rf - confidence_margin)).round(0)
predictions_df['rf_upper_bound'] = (y_pred_rf + confidence_margin).round(0)

print("--- Sample Demand Forecasts with 95% Confidence Ranges ---")
print(predictions_df[['actual_demand', 'random_forest_pred', 'rf_lower_bound', 'rf_upper_bound']].head(6))

# Save finalized evaluation report dataset
predictions_df.to_csv("retail_demand_final_evaluated.csv", index=False)
print("\nSaved finalized evaluated predictions to 'retail_demand_final_evaluated.csv'.")

     DAY 68: MODEL EVALUATION & COMPARISON REPORT

--- Baseline Model (Linear Regression) ---
MAE:  23.85 units
RMSE: 30.77 units
R²:   0.8313

--- Improved Model (Random Forest) ---
MAE:  9.75 units
RMSE: 14.46 units
R²:   0.9627

--- Sample Demand Forecasts with 95% Confidence Ranges ---
   actual_demand  random_forest_pred  rf_lower_bound  rf_upper_bound
0             51               49.61            21.0            78.0
1            230              209.53           181.0           238.0
2            183              180.26           152.0           209.0
3              5               19.88             0.0            48.0
4             83               76.37            48.0           105.0
5              5                5.08             0.0            33.0

Saved finalized evaluated predictions to 'retail_demand_final_evaluated.csv'.


### Guided Question:

Why is evaluating a regression model using multiple metrics (MAE, RMSE, and $R^2$) much more informative than relying on a single metric?

### Answer:

Relying on a single metric to evaluate a regression model creates significant analytical blind spots. In real-world domains like retail demand forecasting, different metrics measure entirely different dimensions of error.Evaluating a model using MAE, RMSE, and $R^2$ simultaneously provides a complete diagnostic profile of model performance.

**1. MAE vs. RMSE: Exposing Outlier Sensitivity**

While both Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE) measure prediction error in the original units of your target variable (e.g., units of product demand), they treat errors mathematically in fundamentally different ways:
- MAE ($\vert{}\text{Actual} - \text{Predicted}\vert{}$): Computes the average magnitude of absolute errors. It treats every error linearly, meaning an error of 50 units is weighted exactly 10 times as heavily as an error of 5 units.
- RMSE ($\sqrt{\frac{1}{n}\sum(\text{Actual} - \text{Predicted})^2}$): Squares the errors before averaging them and taking the square root. This non-linear transformation heavily penalizes large errors.

**Why the comparison matters:**

Comparing MAE and RMSE side-by-side reveals how volatile your model's errors are:
- If $\text{RMSE} \approx \text{MAE}$, your model’s errors are relatively uniform and consistent across the dataset.
- If $\text{RMSE} \gg \text{MAE}$, your model is making occasional, massive miscalculations (e.g., catastrophically underpredicting a major holiday surge). In retail, a model with a great MAE on regular days but a disastrous RMSE during peak shopping seasons will cause severe stockouts and lost revenue.

**2. Absolute Error vs. Relative Variance ($R^2$)**

MAE and RMSE are scale-dependent: an RMSE of 15 units means something entirely different if you are predicting items sold by a massive warehouse (where volume is in the tens of thousands) versus a small local boutique.

The Coefficient of Determination ($R^2$) solves this by offering a scale-free relative measure:
- $R^2$ measures the proportion of variance in the dependent variable that is predictable from your features, scaled against a baseline model that simply predicts the mean demand.
- An $R^2$ of $0.85$ tells you instantly that your feature set explains $85\%$ of the variance in sales, regardless of whether the target units are measured in single items or millions of dollars.

However, $R^2$ alone can be misleading. A high $R^2$ can mask large absolute errors if the overall variance in the dataset is massive.

### Guided Question:

In retail inventory management, what is the practical advantage of providing a confidence range (lower and upper bounds) alongside a point prediction of demand?

### Answer:

In retail inventory management, relying solely on a point prediction (e.g., “We will sell exactly 1,000 units of this item next week”) assumes that demand is deterministic. In reality, retail demand is stochastic, volatile, and influenced by uncaptured external shocks.

Providing a confidence range (or prediction interval, such as an 80% or 95% range of 820 to 1,210 units) alongside the point estimate transforms a raw machine learning output into an actionable operational tool.

**1. Eliminating Stockouts During Demand Surges (The Upper Bound)**

A point prediction represents the expected average outcome (a 50/50 split where actual demand is equally likely to be higher or lower). If a retailer stocks inventory based strictly on a point prediction, they will run out of stock roughly half the time due to normal variance or unexpected holiday surges.
- The Practical Advantage: The upper bound of the prediction interval establishes a reliable ceiling for high-demand scenarios. Supply chain managers can use this upper threshold to size their safety stock, ensuring that even during a bullish demand spike, the store retains enough inventory to capture full revenue without experiencing stockouts.

**2. Controlling Working Capital and Overstock Costs (The Lower Bound)**

Holding excess inventory ties up critical working capital, incurs warehouse storage fees, and risks markdowns or product obsolescence (especially for seasonal or perishable goods).
- The Practical Advantage: The lower bound reveals the baseline minimum demand the retailer is statistically guaranteed to clear. By understanding this floor, procurement teams can make conservative purchasing decisions for low-margin or high-risk SKUs, preventing dead stock accumulation during sluggish economic periods.

**3. Solving the Retail "Newsvendor" Trade-Off**

Inventory planning is fundamentally a Newsvendor Problem: balancing the cost of understocking (lost profit margin and customer dissatisfaction) against the cost of overstocking (holding costs and waste).
- The Practical Advantage: A point prediction gives you no leverage over this financial trade-off. Confidence ranges allow inventory systems to move beyond static rules and adopt quantile-based ordering. For example, a high-margin luxury item might trigger a purchase order tied to the 85th percentile of the prediction range, whereas a low-margin commodity item might anchor closer to the median prediction.

**4. Factoring in Supply Chain Lead Times**

Demand volatility compounds over time. If a supplier takes two weeks to deliver a replenishment order, demand uncertainty multiplies across that entire 14-day window.
- The Practical Advantage: Confidence ranges widen as the forecast horizon extends into the future. By feeding interval bounds into replenishment algorithms, automated inventory systems can dynamically scale safety buffers based on lead-time variance, protecting the store against both delayed shipments and sudden demand acceleration.

### Guided Question:

In the evaluation script above, how did we utilize the standard deviation of the model's residuals to construct the 95% confidence interval?

### Answer:

In the evaluation script, the standard deviation of the model's residuals ($\sigma$) is used to quantify the spread of prediction errors, which is then scaled by a critical z-score to establish the boundary width of the 95% confidence interval.

Here is a step-by-step breakdown of how the script implements this mechanism:

**1. Extracting the Residuals (`y_true - y_pred_rf`)**

The script first computes the residuals by subtracting the Random Forest point predictions from the actual demand values:$$\text{Residuals} = y_{\text{true}} - \hat{y}_{\text{rf}}$$

Each residual represents the exact error (both positive overpredictions and negative underpredictions) that the model made across the test dataset.

**2. Measuring Error Dispersion (`np.std(residuals`))**

The standard deviation of these residuals ($\text{residual\_std}$) measures the overall volatility and spread of the model's errors. A lower standard deviation indicates tight, consistent predictions, while a higher standard deviation signals volatile errors.

**3. Scaling by the 95% Z-Score ($1.96 \times \text{residual\_std}$)**

Assuming the model's errors are approximately normally distributed around zero, statistical theory dictates that roughly 95% of all prediction errors will fall within $\pm 1.96$ standard deviations of the mean.

Multiplying the standard deviation by $1.96$ yields the confidence margin:
- $$\text{Confidence Margin} = 1.96 \times \sigma_{\text{residuals}}$$
- This margin represents the symmetrical radius of uncertainty that must be added to and subtracted from every point prediction.

**4. Constructing Bounded Intervals (`rf_lower_bound` and `rf_upper_bound`)**

Finally, the script applies this margin to each individual point prediction ($\hat{y}_{\text{rf}}$):
- Upper Bound:$$\text{Upper Bound} = \hat{y}_{\text{rf}} + \text{Confidence Margin}$$
- Lower Bound (with a zero floor):$$\text{Lower Bound} = \max\left(0, \hat{y}_{\text{rf}} - \text{Confidence Margin}\right)$$

Using `np.maximum(0, ...)` ensures that the lower bound never dips into negative product demand, which is economically impossible for retail sales inventory planning.